In [1]:
import pandas as pd
import re

complaints = pd.read_csv("data/nhtsa_complaints.csv")

print(complaints.shape)
complaints.head()

(10000, 13)


,complaint_id,make,model,model_year,components,crash,fire,injured,deaths,incident_date,received_date,state,narrative
0,11633486,CFMOTO,450SS,2024.0,SERVICE BRAKES,N,N,0,0,2024-06-11,2025-01-01,GA,Rear brake pedal travels fully to pedal stop w...
1,11633536,KIA,NIRO EV,2020.0,ELECTRICAL SYSTEM; FUEL/PROPULSION SYSTEM; UNK...,N,N,0,0,2024-03-01,2025-01-01,CA,"About 7 months ago, ""Refill Inverter Coolant"" ..."
2,11633525,HONDA,CR-V,2016.0,ENGINE; FUEL/PROPULSION SYSTEM; UNKNOWN OR OTHER,N,N,0,0,2024-12-02,2025-01-01,TX,The engine sporadically will “skip” and then k...
3,11633653,FORD,FOCUS,2016.0,POWER TRAIN,N,N,0,0,2019-01-01,2025-01-02,IN,The contact owns a 2016 Ford Focus. The contac...
4,11633573,RAM,1500,2018.0,ENGINE,N,N,0,0,2024-08-01,2025-01-02,AL,"my vehicle started knocking loudly, the all of..."


# Question 1

In [2]:
# Check for the literal em dash, not a hyphen or an en dash.
# Treat missing narratives as not containing an em dash.
has_em_dash = complaints["narrative"].str.contains(
    "—", regex=False, na=False
)

em_dash_count = has_em_dash.sum()
em_dash_percentage = em_dash_count / len(complaints) * 100

print(f"Narratives with an em dash: {em_dash_count}")
print(f"Percentage: {em_dash_percentage:.2f}%")

Narratives with an em dash: 238
Percentage: 2.38%


In [3]:
# Standardize curly quotation marks and apostrophes.
complaints["text"] = (
    complaints["narrative"]
    .str.replace("“", '"', regex=False)
    .str.replace("”", '"', regex=False)
    .str.replace("‘", "'", regex=False)
    .str.replace("’", "'", regex=False)
)

complaints[["narrative", "text"]].head()

,narrative,text
0,Rear brake pedal travels fully to pedal stop w...,Rear brake pedal travels fully to pedal stop w...
1,"About 7 months ago, ""Refill Inverter Coolant"" ...","About 7 months ago, ""Refill Inverter Coolant"" ..."
2,The engine sporadically will “skip” and then k...,"The engine sporadically will ""skip"" and then k..."
3,The contact owns a 2016 Ford Focus. The contac...,The contact owns a 2016 Ford Focus. The contac...
4,"my vehicle started knocking loudly, the all of...","my vehicle started knocking loudly, the all of..."


The em dash is a punctuation mark that people can use without AI. The 2.38% is the percentage of em dashes used, not the percentage written by AI.

# Question 2

## 2a

In [4]:
# Find a complaint with at least two quoted passages.
for text in complaints["text"].dropna():
    if len(re.findall(r'"(.+?)"', text)) >= 2:
        example = text
        break

print(example)

print("\nGreedy:")
print(re.findall(r'"(.+)"', example))

print("\nLazy:")
print(re.findall(r'"(.+?)"', example))

According to the NHTSA documentation,   "REQUIREMENT: The rear visibility system must default to the rearview image meeting the requirements of S5.5.1 and S5.5.2 (S6.2.1 and S6.2.2) at the beginning of each backing event regardless of any modification to the field of view the driver has previously selected."  The Polestar 3 does show the rearview camera upon initially entering reverse, but if the car is placed in drive and then placed back into reverse, only the top-down "bird-eye" view is displayed.  The driver must manually configure the camera to show the rear view again.  This would appear to be a violation of this requirement.

Greedy:
['REQUIREMENT: The rear visibility system must default to the rearview image meeting the requirements of S5.5.1 and S5.5.2 (S6.2.1 and S6.2.2) at the beginning of each backing event regardless of any modification to the field of view the driver has previously selected."  The Polestar 3 does show the rearview camera upon initially entering reverse, b

The greedy usually captures everything between the first opening quote and the last closing quote, which returns a long passage, including the text between quotations. However, the lazy returns two separate passages.

## 2b

In [16]:
# Match a complete signal word, so "light" does not match "headlight".
dashboard_pattern = (
    r'\b(?:message|messages|warning|light|display|displayed|screen|dash)\b'
    r'[^".!?]*"(.+?)"'
)
# Before the quoted message, do not cross another quote or . ! ?

complaints["dashboard_message"] = complaints["text"].str.extract(
    dashboard_pattern,
    flags=re.IGNORECASE,
    expand=False
)
# Count complaints with an extracted message; missing matches are excluded.
print(
    "Narratives with a dashboard message:",
    complaints["dashboard_message"].count()
)
# Fix capitalization.
# Show the ten most common messages.
print(
    complaints["dashboard_message"]
    .str.lower()
    .value_counts()
    .head(10)
)

Narratives with a dashboard message: 422
dashboard_message
shift to park                                   13
service power steering now                       6
engine overheating                               4
engine malfunction power reduced service now     4
camera not available                             4
power steering assist fault                      3
service transmission                             3
engine power reduced                             3
brake                                            2
check engine                                     2
Name: count, dtype: int64


There are 422 narratives related to the dashboard. Shift to park is the most frequent one with 13 times.

# Question 3

## 3a

In [6]:
# Match dollars
cost_pattern = (
    r'\$\s*'
    r'(?P<amount>\d+(?:,\d{3})*(?:\.\d+)?)'
    r'(?P<thousands>[kK])?'
)

def extract_costs(text):
    """Return all dollar amounts in the text as numbers."""
    costs = []
    # Find every matching amount in the narrative
    for match in re.finditer(cost_pattern, text):
        amount = float(match.group("amount").replace(",", ""))
        # Interpret either k or K as multiplication by 1,000
        if match.group("thousands"):
            amount = amount * 1000

        costs.append(amount)
    # Return an empty list if no dollar amounts were found.
    return costs

In [7]:
test_strings = [
    "The repair cost $2,500.",
    "I paid $1,400.00 for the part",
    "They quoted $3k for a new engine",
    "It will be about $15K",
    "Towing was $350 and diagnosis $200.",
    "No cost mentioned, just 2 visits"
]

for text in test_strings:
    print(extract_costs(text))

[2500.0]
[1400.0]
[3000.0]
[15000.0]
[350.0, 200.0]
[]


In [8]:
complaints["costs"] = [
    extract_costs(text)
    for text in complaints["text"].fillna("")
]

complaints[["complaint_id", "costs"]].head(10)

,complaint_id,costs
0,11633486,[]
1,11633536,[]
2,11633525,[]
3,11633653,[]
4,11633573,[]
5,11633654,[]
6,11633770,[]
7,11633672,[]
8,11633580,[]
9,11633574,[]


In [9]:
has_dollar = complaints["text"].str.contains(
    "$", regex=False, na=False
)

complaints.loc[
    has_dollar, ["complaint_id", "text", "costs"]
].head(10)

,complaint_id,text,costs
38,11633807,The engine blew up while driving down the road...,"[200.0, 8500.0]"
44,11634091,Ball Joint broke on front passenger side 2 tim...,"[6500.0, 2500.0]"
49,11634079,Steering wheel/power steering sticks at a cert...,[300.0]
50,11634248,Engine subframe catastrophic failure (left sid...,[2200.0]
62,11634477,1) Passenger restraint system malfunction. Ac...,"[2700.0, 3000.0]"
64,11634255,The day time running light on the driver side ...,[2100.0]
76,11634731,Both rear Lower control arms are rusted thru a...,[1400.0]
82,11634642,The latch system on the back hatch has a defec...,[1000.0]
83,11634711,"Headlights are so dim, it is dangerous to driv...",[1000.0]
90,11634560,Rear window leaks water into the cab during ra...,[1000.0]


## 3b

In [17]:
# Keep the largest extracted amount for each complaint.
# If no amount was found, leave it missing.
complaints["max_cost"] = [
    max(costs) if costs else None
    for costs in complaints["costs"]
]

# Select the five complaints with the largest amounts.
top5 = complaints.sort_values(
    "max_cost", ascending=False
).head(5)

top5[["complaint_id", "max_cost"]]

,complaint_id,max_cost
3912,11685192,3400000.00
8032,11743170,100000.00
3213,11676141,87821.64
9459,11763253,80000.00
144,11635189,80000.00


In [11]:
#Review the text for the top 5 matches
for index in top5.index:
    text = top5.loc[index, "text"]
    highest = top5.loc[index, "max_cost"]

    print(f"\nComplaint: {top5.loc[index, 'complaint_id']}")
    print(f"Maximum amount: ${highest:,.2f}")
    # Convert each match.
    for match in re.finditer(cost_pattern, text):
        amount = float(match.group("amount").replace(",", ""))

        if match.group("thousands"):
            amount = amount * 1000

        if amount == highest:
            # Include up to 100 characters before and after the match.
            start = max(0, match.start() - 100)
            end = match.end() + 100
            print("Context:", text[start:end])


Complaint: 11685192
Maximum amount: $3,400,000.00
Context: accident. I took it to several mechanics. They say it will cost $1,700k to replace ONE control arm! $3,400k for both. The rubber bushings are completely deteriorated due to the rubber being bad. (My dashboar

Complaint: 11743170
Maximum amount: $100,000.00
Context: on the interstate, not to mention the level of stress and issues this car nightmare has caused me.  $100k vehicle and I still owe 87K

Complaint: 11676141
Maximum amount: $87,821.64
Context: ovided a lesser-value loaner, inadequate for a premium SUV. The vehicle was purchased on [XXX], for $87,821.64 (per Acceptance_Declination Sheet_[XXX}), entitling me to a safe vehicle under warranty. On July 23

Complaint: 11763253
Maximum amount: $80,000.00
Context: creates a very dangerous situation that could cause serious injury or death if the wipers (on a new $80,000 vehicle) suddenly stop while driving in rain. This is a serious danger to public safety and GM need

Compla

The largest extracted amount is 3,400,000, comes from $3,400k, which is likely an typo. Other top amounts describe vehicle values, instead of repair costs.

To better locate the repair costs, we could examine nearby words such as "repair", "costs", "replace", or any natural patterns. Any unusual amount may require manual checking. To capture amounts without $, we could add patterns for a number followed by k or dollars, combined with repair-related wording.

# Question 4

In [18]:
# Capture the full integer according to the question.
speed_pattern = (
    r'\b(\d+)'
    r'(?:\s*(?:-|–|to)\s*\d+)?'
    r'\s*(?:mph|miles\s+per\s+hour)\b'
)

In [13]:
test_speed = pd.Series([
    "Driving at 45 mph",
    "Driving at 45 MPH",
    "Driving at 45mph",
    "Driving at 45 miles per hour",
    "Driving at 55-60 mph",
    "Driving at 55–60 mph",
    "Driving at 55 to 60 mph",
    "Driving at 1045 mph",
    "The car has 45000 miles"
])

print(
    test_speed.str.extract(
        speed_pattern,
        flags=re.IGNORECASE,
        expand=False
    )
)

0      45
1      45
2      45
3      45
4      55
5      55
6      55
7    1045
8     NaN
dtype: str


In [14]:
complaints["speed_text"] = (
    complaints["text"]
    .str.extract(speed_pattern, flags=re.IGNORECASE, expand=False)
    .astype(float)
)

print(
    "Narratives mentioning a speed:",
    complaints["speed_text"].count()
)

Narratives mentioning a speed: 1242


In [15]:
# Export the four required columns, keeping every complaint.
extractions = complaints[
    ["complaint_id", "dashboard_message", "max_cost", "speed_text"]
]

extractions.to_csv("hw05_extractions.csv", index=False)

print(extractions.shape)
extractions.head()

(10000, 4)


,complaint_id,dashboard_message,max_cost,speed_text
0,11633486,NaN,NaN,NaN
1,11633536,NaN,NaN,NaN
2,11633525,NaN,NaN,NaN
3,11633653,NaN,NaN,NaN
4,11633573,NaN,NaN,NaN
